# PulseIQ – AI-Powered Sales Prediction & Business Intelligence Dashboard

Problem Statement:
Businesses generate large volumes of sales and marketing data but often struggle to extract meaningful insights for decision-making. This project aims to develop a system that enables customer segmentation, churn prediction, and revenue forecasting using advanced analytics techniques.

#  Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.arima.model import ARIMA

import warnings
warnings.filterwarnings("ignore")


# 2 Load Dataset

In [3]:
df = pd.read_csv("data/Superstore.csv",encoding='latin1')

print("Shape:", df.shape)
df.head()

Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


# 3 Data Cleaning

In [ ]:
df.drop_duplicates(inplace=True)

df['Order Date'] = pd.to_datetime(df['Order Date'])

In [ ]:
# Feature Engineering

df['Month'] = df['Order Date'].dt.month
df['Year'] = df['Order Date'].dt.year


# 4 EDA(Exploaratory Data Analysis)

In [ ]:
print("\nMissing Values:\n", df.isnull().sum())

plt.figure()
sns.heatmap(df.corr(numeric_only=True), annot=True)
plt.title("Correlation Heatmap")
plt.show()


#  Customer Segmentation using KMeans Clustering with RFM Features

In [ ]:
rfm = df.groupby('Customer Name').agg({
    'Sales': 'sum',
    'Quantity': 'sum',
    'Order Date': 'max'
})

rfm['Recency'] = (df['Order Date'].max() - rfm['Order Date']).dt.days

rfm = rfm[['Sales','Quantity','Recency']]

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

# Finding Optimal Clusters using Elbow Method

In [ ]:
inertia = []
K = range(1,10)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(rfm_scaled)
    inertia.append(kmeans.inertia_)

plt.plot(K, inertia)
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

# Customer Segmentation Visualization

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

plt.figure()
sns.scatterplot(x=rfm['Sales'], y=rfm['Recency'], hue=rfm['Cluster'])
plt.title("Customer Segmentation")
plt.show()

# Customer Churn Prediction using XGBoost

In [ ]:
cust_df = df.groupby('Customer Name').agg({
    'Sales':'sum',
    'Quantity':'sum',
    'Discount':'mean',
    'Order Date':'max'
})

# feature Engineering for churn Prediction
cust_df['LastPurchaseDays'] = (df['Order Date'].max() - cust_df['Order Date']).dt.days
cust_df['Churn'] = (cust_df['LastPurchaseDays'] > 90).astype(int)

# Features & target
X = cust_df[['Sales','Quantity','Discount']]
y = cust_df['Churn']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Training - XGBoost Classifier

In [ ]:
model = XGBClassifier()
model.fit(X_train, y_train)

# Predictions
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:,1]

# Evaluation
print("\nClassification Report:\n", classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, proba))

# Sales Prediction using Random Forest Regressor

In [ ]:
df = df[['Sales', 'Profit', 'Quantity', 'Discount']].dropna()

X = df[['Profit', 'Quantity', 'Discount']]
y = df['Sales']

model = RandomForestRegressor()
model.fit(X, y)

#  Model Explainability using SHAP

In [ ]:
import shap

# Create explainer
explainer = shap.Explainer(model)

# Calculate SHAP values
shap_values = explainer(X_test)

# Summary Plot (Most Important Features)
shap.summary_plot(shap_values, X_test)

# Bar Plot (Feature Importance)
shap.plots.bar(shap_values)

# Force Plot (Single Prediction Explain)
shap.initjs()
shap.plots.force(shap_values[0])

# Revenue Forecasting using ARIMA Time Series Model

In [ ]:
ts = df.groupby('Order Date')['Sales'].sum()

model_arima = ARIMA(ts, order=(5,1,0))
model_arima_fit = model_arima.fit()

forecast = model_arima_fit.forecast(steps=30)

plt.figure()
ts.plot(label='Actual')
forecast.plot(label='Forecast')
plt.legend()
plt.title("Sales Forecast")
plt.show()


# Final Insights

In [ ]:
print("\n===== FINAL INSIGHTS =====")

print("Top Category:", df.groupby('Category')['Sales'].sum().idxmax())
print("Top Region:", df.groupby('Region')['Profit'].sum().idxmax())
print("Best Customer:", df.groupby('Customer Name')['Sales'].sum().idxmax())

In [ ]:
# SAVE MODEL USING JOBLIB

import joblib
import os

# Create folder
os.makedirs("models", exist_ok=True)

# Save model
joblib.dump(model, "models/churn_model.pkl")

# Save scaler
joblib.dump(scaler, "models/scaler.pkl")

print("✅ Model & Scaler saved successfully!")

# Conclusion

In [ ]:
- Performed Customer Segmentation using KMeans  
- Built Churn Prediction Model using XGBoost  
- Forecasted Revenue using ARIMA  
- Used SHAP for model explainability  

This project demonstrates end-to-end Data Science pipeline